# 08 — Unsupervised Machine Learning


In [1]:
#Imports

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
)
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

ROOT = Path.cwd().parent
DATA_FILE = ROOT / "artifacts" / "cleaned_data.parquet"
OUTPUT_DIR = ROOT / "artifacts" / "unsupervised"
CHART_DIR = ROOT / "artifacts" / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
#Load data

df = pd.read_parquet(DATA_FILE)

TARGET_COLUMN = "purchased"
ID_COLUMNS = ["customer_id"]

excluded_columns = [
    column
    for column in [TARGET_COLUMN] + ID_COLUMNS
    if column in df.columns
]

X_unsupervised = df.drop(
    columns=excluded_columns
).copy()

print("Original shape:", df.shape)
print("Unsupervised feature shape:", X_unsupervised.shape)
print("Excluded columns:", excluded_columns)
print("Features:", X_unsupervised.columns.tolist())

display(X_unsupervised.head())

Original shape: (500, 7)
Unsupervised feature shape: (500, 5)
Excluded columns: ['purchased', 'customer_id']
Features: ['age', 'income', 'region', 'visits', 'satisfaction']


,age,income,region,visits,satisfaction
0,58,47184,North,9,2.0
1,65,44868,North,19,2.7
2,23,75314,South,17,3.4
3,63,127974,South,15,3.4
4,66,59853,West,11,2.1


In [8]:
#Feature type detection

numeric_features = X_unsupervised.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_unsupervised.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


Numerical features: ['age', 'income', 'visits', 'satisfaction']
Categorical features: ['region']


In [9]:
#Unsupervised processor

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

unsupervised_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)

X_processed = unsupervised_preprocessor.fit_transform(
    X_unsupervised
)

feature_names = (
    unsupervised_preprocessor.get_feature_names_out()
)

print("Processed shape:", X_processed.shape)
print("Missing values:", np.isnan(X_processed).sum())
print("Infinite values:", np.isinf(X_processed).sum())

print("\nProcessed features:")

for feature_name in feature_names:
    print(feature_name)

Processed shape: (500, 8)
Missing values: 0
Infinite values: 0

Processed features:
numeric__age
numeric__income
numeric__visits
numeric__satisfaction
categorical__region_East
categorical__region_North
categorical__region_South
categorical__region_West


In [10]:
#Find best k means cluster count

maximum_clusters = min(10, len(X_processed) - 1)

kmeans_evaluation = []

for number_of_clusters in range(2, maximum_clusters + 1):
    model = KMeans(
        n_clusters=number_of_clusters,
        n_init=20,
        random_state=RANDOM_STATE,
    )

    labels = model.fit_predict(X_processed)

    kmeans_evaluation.append({
        "number_of_clusters": number_of_clusters,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(
            X_processed,
            labels,
        ),
        "davies_bouldin_score":
            davies_bouldin_score(
                X_processed,
                labels,
            ),
        "calinski_harabasz_score":
            calinski_harabasz_score(
                X_processed,
                labels,
            ),
    })

kmeans_evaluation_df = pd.DataFrame(
    kmeans_evaluation
)

display(kmeans_evaluation_df.round(4))

,number_of_clusters,inertia,silhouette_score,davies_bouldin_score,calinski_harabasz_score
0,2,1981.2429,0.1576,2.1947,98.8802
1,3,1731.6717,0.1495,1.8879,92.2661
2,4,1538.0670,0.1543,1.7198,89.9255
3,5,1404.5184,0.1581,1.6155,85.4749
4,6,1300.0267,0.1579,1.5513,81.6680
5,7,1202.0054,0.1660,1.5214,80.1581
6,8,1109.9047,0.1771,1.3762,80.0897
7,9,1057.1644,0.1688,1.4353,76.4870
8,10,1017.0151,0.1610,1.4619,72.6779


In [11]:
#clusterselection - metrics plot

fig = px.line(
    kmeans_evaluation_df,
    x="number_of_clusters",
    y="silhouette_score",
    markers=True,
    title="K-Means Silhouette Score",
)

fig.show()
fig.write_html(
    CHART_DIR / "kmeans_silhouette_scores.html"
)

fig = px.line(
    kmeans_evaluation_df,
    x="number_of_clusters",
    y="inertia",
    markers=True,
    title="K-Means Elbow Curve",
)

fig.show()
fig.write_html(
    CHART_DIR / "kmeans_elbow_curve.html"
)

In [12]:
# Train the best k means model

BEST_K = int(
    kmeans_evaluation_df.loc[
        kmeans_evaluation_df[
            "silhouette_score"
        ].idxmax(),
        "number_of_clusters",
    ]
)

best_kmeans_model = KMeans(
    n_clusters=BEST_K,
    n_init=20,
    random_state=RANDOM_STATE,
)

kmeans_labels = best_kmeans_model.fit_predict(
    X_processed
)

print("Selected number of clusters:", BEST_K)

print(
    "Silhouette score:",
    round(
        silhouette_score(
            X_processed,
            kmeans_labels,
        ),
        4,
    ),
)

Selected number of clusters: 8
Silhouette score: 0.1771


In [13]:
#PCA(Dimensionality Reduction)

pca_model = PCA(
    n_components=2,
    random_state=RANDOM_STATE,
)

pca_coordinates = pca_model.fit_transform(
    X_processed
)

pca_df = pd.DataFrame({
    "principal_component_1":
        pca_coordinates[:, 0],
    "principal_component_2":
        pca_coordinates[:, 1],
    "cluster": kmeans_labels.astype(str),
})

explained_variance = (
    pca_model.explained_variance_ratio_
)

print(
    "PC1 explained variance:",
    round(explained_variance[0], 4),
)

print(
    "PC2 explained variance:",
    round(explained_variance[1], 4),
)

print(
    "Total explained variance:",
    round(explained_variance.sum(), 4),
)

PC1 explained variance: 0.2335
PC2 explained variance: 0.2122
Total explained variance: 0.4457


In [14]:
#Cluster Visuvalization

fig = px.scatter(
    pca_df,
    x="principal_component_1",
    y="principal_component_2",
    color="cluster",
    opacity=0.75,
    title=(
        f"K-Means Customer Segments "
        f"(K = {BEST_K})"
    ),
)

fig.show()

fig.write_html(
    CHART_DIR / "kmeans_pca_clusters.html"
)

In [15]:
#Attach clusters to original data

clustered_df = df.copy()

clustered_df["kmeans_cluster"] = kmeans_labels

display(clustered_df.head())

,customer_id,age,income,region,visits,satisfaction,purchased,kmeans_cluster
0,1,58,47184,North,9,2.0,0,6
1,2,65,44868,North,19,2.7,0,7
2,3,23,75314,South,17,3.4,1,0
3,4,63,127974,South,15,3.4,1,3
4,5,66,59853,West,11,2.1,0,6


In [16]:
#Cluster Sizes

cluster_sizes = (
    clustered_df["kmeans_cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="customers")
)

cluster_sizes["percentage"] = (
    cluster_sizes["customers"]
    / len(clustered_df)
    * 100
).round(2)

display(cluster_sizes)

,cluster,customers,percentage
0,0,64,12.8
1,1,63,12.6
2,2,60,12.0
3,3,69,13.8
4,4,50,10.0
5,5,65,13.0
6,6,65,13.0
7,7,64,12.8


In [17]:
#Business Cluster Profiles

numeric_profile_columns = [
    column
    for column in numeric_features
    if column in clustered_df.columns
]

cluster_profile = (
    clustered_df.groupby("kmeans_cluster")
    .agg(
        customers=("kmeans_cluster", "size"),
        average_age=("age", "mean"),
        average_income=("income", "mean"),
        average_visits=("visits", "mean"),
        average_satisfaction=("satisfaction", "mean"),
        purchase_rate=(TARGET_COLUMN, "mean"),
    )
    .reset_index()
)

cluster_profile["purchase_rate_percentage"] = (
    cluster_profile["purchase_rate"] * 100
)

display(cluster_profile.round(2))

,kmeans_cluster,customers,average_age,average_income,average_visits,average_satisfaction,purchase_rate,purchase_rate_percentage
0,0,64,30.80,49976.16,15.23,2.50,0.50,50.00
1,1,63,54.67,116521.76,5.38,3.76,0.73,73.02
2,2,60,29.65,66035.60,5.73,4.20,0.57,56.67
3,3,69,53.28,115869.06,15.13,2.08,0.91,91.30
4,4,50,32.68,117110.48,5.40,1.88,0.38,38.00
5,5,65,34.82,115779.97,15.06,4.02,1.00,100.00
6,6,65,54.43,55401.15,5.88,1.91,0.11,10.77
7,7,64,56.77,53036.14,13.27,3.90,0.59,59.38


In [18]:
#REgion Distribution inside clusters

region_cluster_table = pd.crosstab(
    clustered_df["kmeans_cluster"],
    clustered_df["region"],
    normalize="index",
).mul(100)

display(region_cluster_table.round(2))

region,East,North,South,West
kmeans_cluster,,,,
0,26.56,21.88,37.50,14.06
1,31.75,28.57,22.22,17.46
2,26.67,26.67,18.33,28.33
3,21.74,15.94,27.54,34.78
4,26.00,26.00,26.00,22.00
5,18.46,38.46,12.31,30.77
6,24.62,21.54,21.54,32.31
7,28.12,20.31,18.75,32.81


In [19]:
#Hierarchical Clustering

hierarchical_model = AgglomerativeClustering(
    n_clusters=BEST_K
)

hierarchical_labels = (
    hierarchical_model.fit_predict(X_processed)
)

hierarchical_silhouette = silhouette_score(
    X_processed,
    hierarchical_labels,
)

print(
    "Hierarchical silhouette score:",
    round(hierarchical_silhouette, 4),
)

clustered_df["hierarchical_cluster"] = (
    hierarchical_labels
)

Hierarchical silhouette score: 0.115


In [20]:
#DB Scan Clustering

dbscan_model = DBSCAN(
    eps=0.8,
    min_samples=8,
)

dbscan_labels = dbscan_model.fit_predict(
    X_processed
)

unique_dbscan_labels = np.unique(dbscan_labels)

dbscan_cluster_count = len(
    unique_dbscan_labels[
        unique_dbscan_labels != -1
    ]
)

dbscan_noise_count = int(
    np.sum(dbscan_labels == -1)
)

print("DBSCAN clusters:", dbscan_cluster_count)
print("DBSCAN noise records:", dbscan_noise_count)
print("DBSCAN labels:", unique_dbscan_labels)

clustered_df["dbscan_cluster"] = dbscan_labels

DBSCAN clusters: 0
DBSCAN noise records: 500
DBSCAN labels: [-1]


In [21]:
#Isolation Forest Anamoly Detection

CONTAMINATION = 0.05

isolation_forest_model = IsolationForest(
    n_estimators=300,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

anomaly_labels = isolation_forest_model.fit_predict(
    X_processed
)

anomaly_scores = (
    isolation_forest_model.decision_function(
        X_processed
    )
)

clustered_df["anomaly_label"] = np.where(
    anomaly_labels == -1,
    "Anomaly",
    "Normal",
)

clustered_df["anomaly_score"] = anomaly_scores

print(
    clustered_df["anomaly_label"].value_counts()
)

print(
    "\nAnomaly percentage:",
    round(
        (
            clustered_df["anomaly_label"]
            .eq("Anomaly")
            .mean()
            * 100
        ),
        2,
    ),
)

anomaly_label
Normal     475
Anomaly     25
Name: count, dtype: int64

Anomaly percentage: 5.0


In [22]:
#Anamoly Inspection

anomalies_df = (
    clustered_df.loc[
        clustered_df["anomaly_label"] == "Anomaly"
    ]
    .sort_values("anomaly_score")
)

display(anomalies_df.head(20))

,customer_id,age,income,region,visits,satisfaction,purchased,kmeans_cluster,hierarchical_cluster,dbscan_cluster,anomaly_label,anomaly_score
388,389,21,144669,West,1,1.0,1,4,0,-1,Anomaly,-0.045136
389,390,18,22384,East,18,4.9,1,0,3,-1,Anomaly,-0.037748
147,148,70,18672,East,10,4.3,1,7,1,-1,Anomaly,-0.018127
377,378,65,143546,South,20,3.8,1,3,2,-1,Anomaly,-0.018116
87,88,66,117714,West,1,4.8,1,1,5,-1,Anomaly,-0.015188
194,195,22,39178,East,20,1.6,0,0,3,-1,Anomaly,-0.014805
137,138,70,27619,West,3,2.3,0,6,1,-1,Anomaly,-0.013124
337,338,20,20429,East,19,3.4,0,0,3,-1,Anomaly,-0.012997
116,117,18,51182,East,2,4.9,0,2,4,-1,Anomaly,-0.012767
276,277,19,55428,West,2,1.5,0,4,3,-1,Anomaly,-0.012192


In [23]:
#Compare Clustering Algorithms

algorithm_comparison = pd.DataFrame([
    {
        "algorithm": "K-Means",
        "clusters": len(np.unique(kmeans_labels)),
        "silhouette_score": silhouette_score(
            X_processed,
            kmeans_labels,
        ),
    },
    {
        "algorithm": "Hierarchical",
        "clusters": len(
            np.unique(hierarchical_labels)
        ),
        "silhouette_score":
            hierarchical_silhouette,
    },
])

if (
    dbscan_cluster_count >= 2
    and dbscan_noise_count < len(X_processed)
):
    non_noise_mask = dbscan_labels != -1

    if len(
        np.unique(
            dbscan_labels[non_noise_mask]
        )
    ) >= 2:
        algorithm_comparison.loc[
            len(algorithm_comparison)
        ] = {
            "algorithm": "DBSCAN",
            "clusters": dbscan_cluster_count,
            "silhouette_score":
                silhouette_score(
                    X_processed[non_noise_mask],
                    dbscan_labels[non_noise_mask],
                ),
        }

display(
    algorithm_comparison.sort_values(
        "silhouette_score",
        ascending=False,
    ).round(4)
)

,algorithm,clusters,silhouette_score
0,K-Means,8,0.1771
1,Hierarchical,8,0.1150


In [24]:
# Automated insights Generation

highest_purchase_cluster = (
    cluster_profile.loc[
        cluster_profile[
            "purchase_rate_percentage"
        ].idxmax()
    ]
)

lowest_purchase_cluster = (
    cluster_profile.loc[
        cluster_profile[
            "purchase_rate_percentage"
        ].idxmin()
    ]
)

unsupervised_insights = [
    (
        f"K-Means selected {BEST_K} clusters using "
        f"the highest silhouette score."
    ),
    (
        f"Cluster {int(highest_purchase_cluster['kmeans_cluster'])} "
        f"has the highest purchase rate at "
        f"{highest_purchase_cluster['purchase_rate_percentage']:.2f}%."
    ),
    (
        f"Cluster {int(lowest_purchase_cluster['kmeans_cluster'])} "
        f"has the lowest purchase rate at "
        f"{lowest_purchase_cluster['purchase_rate_percentage']:.2f}%."
    ),
    (
        f"Isolation Forest marked "
        f"{len(anomalies_df)} records as potential anomalies."
    ),
    (
        "Cluster labels describe statistical similarity. "
        "They do not automatically represent causal "
        "customer categories."
    ),
]

for number, insight in enumerate(
    unsupervised_insights,
    start=1,
):
    print(f"{number}. {insight}")

1. K-Means selected 8 clusters using the highest silhouette score.
2. Cluster 5 has the highest purchase rate at 100.00%.
3. Cluster 6 has the lowest purchase rate at 10.77%.
4. Isolation Forest marked 25 records as potential anomalies.
5. Cluster labels describe statistical similarity. They do not automatically represent causal customer categories.


In [25]:
# Save outputs

joblib.dump(
    unsupervised_preprocessor,
    OUTPUT_DIR / "unsupervised_preprocessor.joblib",
)

joblib.dump(
    best_kmeans_model,
    OUTPUT_DIR / "kmeans_model.joblib",
)

joblib.dump(
    pca_model,
    OUTPUT_DIR / "pca_model.joblib",
)

joblib.dump(
    isolation_forest_model,
    OUTPUT_DIR / "isolation_forest_model.joblib",
)

clustered_df.to_parquet(
    OUTPUT_DIR / "clustered_data.parquet",
    index=False,
)

cluster_profile.to_csv(
    OUTPUT_DIR / "cluster_profiles.csv",
    index=False,
)

kmeans_evaluation_df.to_csv(
    OUTPUT_DIR / "kmeans_evaluation.csv",
    index=False,
)

algorithm_comparison.to_csv(
    OUTPUT_DIR / "algorithm_comparison.csv",
    index=False,
)

anomalies_df.to_csv(
    OUTPUT_DIR / "anomalies.csv",
    index=False,
)

unsupervised_summary = {
    "selected_k": BEST_K,
    "kmeans_silhouette_score": float(
        silhouette_score(
            X_processed,
            kmeans_labels,
        )
    ),
    "hierarchical_silhouette_score": float(
        hierarchical_silhouette
    ),
    "dbscan_clusters": int(dbscan_cluster_count),
    "dbscan_noise_records": int(dbscan_noise_count),
    "anomaly_records": int(len(anomalies_df)),
    "anomaly_percentage": float(
        len(anomalies_df) / len(clustered_df) * 100
    ),
    "pca_explained_variance": float(
        explained_variance.sum()
    ),
    "insights": unsupervised_insights,
}

with open(
    OUTPUT_DIR / "unsupervised_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        unsupervised_summary,
        file,
        indent=4,
    )

print("Unsupervised outputs saved successfully.")

Unsupervised outputs saved successfully.


In [26]:
#VErify Saved files

required_files = [
    "unsupervised_preprocessor.joblib",
    "kmeans_model.joblib",
    "pca_model.joblib",
    "isolation_forest_model.joblib",
    "clustered_data.parquet",
    "cluster_profiles.csv",
    "kmeans_evaluation.csv",
    "algorithm_comparison.csv",
    "anomalies.csv",
    "unsupervised_summary.json",
]

for filename in required_files:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {path.exists()}")

unsupervised_preprocessor.joblib: True
kmeans_model.joblib: True
pca_model.joblib: True
isolation_forest_model.joblib: True
clustered_data.parquet: True
cluster_profiles.csv: True
kmeans_evaluation.csv: True
algorithm_comparison.csv: True
anomalies.csv: True
unsupervised_summary.json: True
